# NLP — Manipulation des fichiers PDF avec PyPDF2

**Objectifs d’apprentissage**

À la fin de ce notebook, vous saurez :
1. ouvrir un fichier PDF et lire ses pages ;
2. compter les pages et accéder à une page précise ;
3. extraire le texte d’une page ou de tout le document ;
4. créer un nouveau PDF à partir de pages sélectionnées ;
5. fusionner plusieurs fichiers PDF.

> **Avant de commencer :** placez `US_Declaration.pdf`, `F1.pdf` et `F2.pdf` dans le même dossier que ce notebook. Les exemples utilisant `F1.pdf` et `F2.pdf` supposent que ces fichiers existent et contiennent suffisamment de pages.


## 1. Installer et importer PyPDF2

Dans Jupyter ou Google Colab, `%pip` installe le paquet dans l’environnement du notebook. Si PyPDF2 est déjà installé, vous pouvez passer à la cellule suivante.

In [ ]:
%pip install -q PyPDF2

In [ ]:
import PyPDF2
from pathlib import Path

# Afficher la version installée
print("Version de PyPDF2 :", PyPDF2.__version__)

## 2. Ouvrir un PDF et compter ses pages

- `open(..., "rb")` ouvre le fichier en **mode lecture binaire**.
- `PdfReader(f)` crée un objet lecteur qui permet d’accéder aux pages.
- `len(pdf_reader.pages)` retourne le nombre total de pages.
- L’utilisation de `with` ferme automatiquement le fichier à la fin du bloc.

**Important :** `PdfFileReader`, `getPage()` et `numPages` appartiennent à l’ancienne API. Dans ce notebook, nous utilisons la syntaxe moderne.

In [ ]:
pdf_path = Path("US_Declaration.pdf")

# Vérifier que le fichier se trouve dans le dossier courant
if not pdf_path.exists():
    raise FileNotFoundError(
        f"Fichier introuvable : {pdf_path}. "
        "Placez US_Declaration.pdf dans le dossier du notebook."
    )

with pdf_path.open("rb") as f:
    pdf_reader = PyPDF2.PdfReader(f)

    # Afficher le nombre total de pages
    print("Nombre de pages :", len(pdf_reader.pages))

## 3. Accéder aux pages du PDF

Les pages sont indexées à partir de **0** :
- `pages[0]` : première page ;
- `pages[1]` : deuxième page ;
- `pages[-1]` : dernière page.

La cellule suivante affiche le nombre de pages et l’objet correspondant à la première page.

In [ ]:
with open("US_Declaration.pdf", "rb") as f:
    pdf_reader = PyPDF2.PdfReader(f)

    # Afficher le nombre de pages
    print("Nombre de pages :", len(pdf_reader.pages))

    # Accéder à la première page si le document n'est pas vide
    if len(pdf_reader.pages) > 0:
        first_page = pdf_reader.pages[0]
        print("Première page :", first_page)
    else:
        print("Le PDF ne contient aucune page.")

## 4. Extraire le texte d’une page

La méthode `extract_text()` tente d’extraire le texte présent dans la page. Le résultat peut être vide si le PDF est composé d’images scannées ou si son texte n’est pas extractible correctement.

Pour un PDF scanné, une étape d’OCR (*Optical Character Recognition*) peut être nécessaire.

In [ ]:
with open("US_Declaration.pdf", "rb") as f:
    pdf_reader = PyPDF2.PdfReader(f)

    if len(pdf_reader.pages) > 0:
        # Récupérer la première page
        first_page = pdf_reader.pages[0]

        # Extraire et afficher son texte
        first_page_text = first_page.extract_text() or ""
        print(first_page_text)
    else:
        print("Le PDF ne contient aucune page.")

### 4.1 Découper le texte en mots

La méthode `.split()` découpe une chaîne de caractères en utilisant les espaces comme séparateurs par défaut. Ce n’est pas une tokenisation linguistique avancée : la ponctuation reste généralement attachée aux mots.

In [ ]:
with open("US_Declaration.pdf", "rb") as f:
    pdf_reader = PyPDF2.PdfReader(f)

    if len(pdf_reader.pages) > 0:
        text = pdf_reader.pages[0].extract_text() or ""

        # Découper le texte selon les espaces
        words = text.split()

        print("Nombre d'éléments obtenus :", len(words))
        print(words[:30])  # Afficher au maximum les 30 premiers éléments

## 5. Extraire le texte de toutes les pages

Nous allons stocker le texte de chaque page dans une liste Python. Ici, nous **n’ajoutons pas de placeholder** : la première page est stockée à l’index `0`, comme dans les listes Python habituelles.

In [ ]:
pdf_text = []

with open("US_Declaration.pdf", "rb") as f:
    pdf_reader = PyPDF2.PdfReader(f)

    # Parcourir les pages dans l'ordre
    for page_number, page in enumerate(pdf_reader.pages, start=1):
        # Extraire le texte ; utiliser une chaîne vide si aucun texte n'est disponible
        page_text = page.extract_text() or ""
        pdf_text.append(page_text)

        print(f"Page {page_number} : {len(page_text)} caractères extraits")

# Afficher le texte de la première page, si elle existe
if pdf_text:
    print(pdf_text[0])

**À retenir :** `pdf_text[0]` contient le texte de la page 1, `pdf_text[1]` celui de la page 2, etc. Le numéro affiché par `enumerate(..., start=1)` commence à 1 uniquement pour rendre le message plus lisible.

## 6. Créer un nouveau PDF avec une page sélectionnée

`PdfWriter()` permet de construire un nouveau document PDF. La méthode `add_page()` ajoute une page au document de sortie et `write()` enregistre le résultat.

Dans cet exemple, nous copions la première page de `US_Declaration.pdf` dans `first_page.pdf`.

In [ ]:
input_path = Path("US_Declaration.pdf")
output_path = Path("first_page.pdf")

with input_path.open("rb") as input_file:
    reader = PyPDF2.PdfReader(input_file)

    if len(reader.pages) == 0:
        raise ValueError("Le PDF source ne contient aucune page.")

    writer = PyPDF2.PdfWriter()

    # Ajouter la première page (index 0) au nouveau document
    writer.add_page(reader.pages[0])

    # Enregistrer le PDF créé
    with output_path.open("wb") as output_file:
        writer.write(output_file)

print(f"PDF créé : {output_path}")

## 7. Fusionner deux fichiers PDF

La fusion consiste à placer les pages de plusieurs fichiers dans un seul document. Ici, nous ajoutons d’abord toutes les pages de `F1.pdf`, puis toutes celles de `F2.pdf`.

**Vérifiez** que les deux fichiers sont dans le dossier du notebook.

In [ ]:
from pathlib import Path
import PyPDF2

filename1 = Path("F1.pdf")
filename2 = Path("F2.pdf")
merged_path = Path("merged_file_1.pdf")

# Vérifier la présence des fichiers d'entrée
for path in (filename1, filename2):
    if not path.exists():
        raise FileNotFoundError(f"Fichier introuvable : {path}")

writer = PyPDF2.PdfWriter()

# Ajouter les pages de F1 puis celles de F2
with filename1.open("rb") as file1, filename2.open("rb") as file2:
    reader1 = PyPDF2.PdfReader(file1)
    reader2 = PyPDF2.PdfReader(file2)

    writer.append(reader1)
    writer.append(reader2)

    # Enregistrer le document fusionné
    with merged_path.open("wb") as output_file:
        writer.write(output_file)

print(f"Fusion terminée : {merged_path}")

## 8. Construire un PDF à partir de pages choisies dans deux fichiers

Les indices Python commencent à 0. Ainsi :
- l’index `5` désigne la **6e page** ;
- l’index `3` désigne la **4e page**.

Le code vérifie d’abord que les pages demandées existent, afin d’éviter une erreur `IndexError`.

In [ ]:
filename1 = Path("F1.pdf")
filename2 = Path("F2.pdf")
output_path = Path("merged_file_2.pdf")

for path in (filename1, filename2):
    if not path.exists():
        raise FileNotFoundError(f"Fichier introuvable : {path}")

with filename1.open("rb") as f1, filename2.open("rb") as f2:
    reader1 = PyPDF2.PdfReader(f1)
    reader2 = PyPDF2.PdfReader(f2)

    # Vérifier que les 6e et 4e pages existent
    if len(reader1.pages) <= 5:
        raise ValueError("F1.pdf doit contenir au moins 6 pages.")
    if len(reader2.pages) <= 3:
        raise ValueError("F2.pdf doit contenir au moins 4 pages.")

    writer = PyPDF2.PdfWriter()

    # Ajouter la 6e page de F1.pdf
    writer.add_page(reader1.pages[5])

    # Ajouter la 4e page de F2.pdf
    writer.add_page(reader2.pages[3])

    # Enregistrer les deux pages dans un nouveau PDF
    with output_path.open("wb") as output_file:
        writer.write(output_file)

print(f"PDF créé : {output_path}")

## 9. Résumé des méthodes utiles

| Méthode / propriété | Rôle |
|---|---|
| `PyPDF2.PdfReader(f)` | Lire un fichier PDF |
| `pdf_reader.pages` | Accéder à la collection des pages |
| `len(pdf_reader.pages)` | Compter les pages |
| `pdf_reader.pages[0]` | Accéder à la première page |
| `page.extract_text()` | Extraire le texte d’une page |
| `PyPDF2.PdfWriter()` | Construire un nouveau PDF |
| `writer.add_page(page)` | Ajouter une page au document de sortie |
| `writer.append(reader)` | Ajouter toutes les pages d’un document |
| `writer.write(file)` | Écrire le document dans un fichier |

## 10. Exercices

1. Affichez le nombre de pages de `F1.pdf`.
2. Extrayez et affichez le texte de la dernière page de `US_Declaration.pdf`.
3. Créez un PDF contenant uniquement les deux premières pages de `US_Declaration.pdf`.
4. Fusionnez `F2.pdf` puis `F1.pdf`. Comparez l’ordre des pages avec celui de la section 7.
5. Modifiez la section 8 pour sélectionner d’autres pages. Vérifiez les indices avant d’exécuter le code.

### Questions de révision

- Pourquoi ouvre-t-on un PDF avec le mode `"rb"` ?
- Pourquoi l’index de la première page est-il `0` ?
- Quelle différence y a-t-il entre `PdfReader` et `PdfWriter` ?
- Pourquoi `extract_text()` peut-il renvoyer une chaîne vide ?

> **Limite importante :** PyPDF2 extrait le texte déjà présent dans le PDF ; il ne reconnaît pas automatiquement le texte contenu dans une image scannée. Pour cela, il faut utiliser un outil OCR.
